# Notebook 05 — Model Training & Comparison

**Goal:** Train multiple regression models on the processed California Housing data, compare them on standardised metrics, and produce presentation-ready charts.

Models:
| # | Model | Notes |
|---|-------|-------|
| 1 | Linear Regression | Baseline linear |
| 2 | Ridge Regression | L2-regularised linear |
| 3 | Lasso Regression | L1-regularised / sparse |
| 4 | Decision Tree | Non-linear, interpretable |
| 5 | Random Forest | Ensemble of trees |
| 6 | Gradient Boosting | Sequential ensemble (best expected) |
| 7 | KNN Regressor | Distance-based baseline |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model  import LinearRegression, Ridge, Lasso
from sklearn.tree          import DecisionTreeRegressor
from sklearn.ensemble      import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors     import KNeighborsRegressor
from sklearn.model_selection import cross_val_score, KFold, learning_curve
from sklearn.metrics       import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib
import os

FIG_DPI  = 150
TITLE_FS = 14
LABEL_FS = 12
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': FIG_DPI, 'savefig.bbox': 'tight'})

# ── Load processed data ─────────────────────────────────────────────────────
X_train = np.load('../../data/processed/X_train.npy')
y_train = np.load('../../data/processed/y_train.npy')
X_test  = np.load('../../data/processed/X_test.npy')
y_test  = np.load('../../data/processed/y_test.npy')

# y is already scaled by /100000; convert back for human-readable RMSE
SCALE   = 100_000.0
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'y_train range: [{y_train.min():.2f}, {y_train.max():.2f}] (×{SCALE:.0f} = USD)')

---
## 1 · Define & Train All Models

In [ ]:
models = {
    'Linear Regression'    : LinearRegression(),
    'Ridge'                : Ridge(alpha=1.0),
    'Lasso'                : Lasso(alpha=0.001, max_iter=5000),
    'Decision Tree'        : DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest'        : RandomForestRegressor(n_estimators=200, max_depth=15,
                                                   random_state=42, n_jobs=-1),
    'Gradient Boosting'    : GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                                       max_depth=5, random_state=42),
    'KNN'                  : KNeighborsRegressor(n_neighbors=10, n_jobs=-1),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred  = model.predict(X_test)
    rmse    = np.sqrt(mean_squared_error(y_test, y_pred)) * SCALE
    mae     = mean_absolute_error(y_test, y_pred) * SCALE
    r2      = r2_score(y_test, y_pred)

    cv_scores = cross_val_score(model, X_train, y_train,
                                cv=kf, scoring='r2', n_jobs=-1)

    results[name] = {
        'RMSE ($)' : rmse,
        'MAE ($)'  : mae,
        'R²'       : r2,
        'CV R² mean': cv_scores.mean(),
        'CV R² std' : cv_scores.std(),
        'Train Time (s)': train_time,
        'model'    : model,
        'y_pred'   : y_pred
    }
    print(f'{name:<22}  RMSE=${rmse:>8,.0f}  MAE=${mae:>7,.0f}  R²={r2:.3f}  '
          f'CV={cv_scores.mean():.3f}±{cv_scores.std():.3f}  t={train_time:.1f}s')

---
### 📊 PLOT 1 — Model Comparison: RMSE, MAE, R² (three-panel bar chart)
**What it shows:** Side-by-side comparison of all models on three metrics.  
**Presentation use:** The definitive model comparison slide — clearly shows ensemble models outperform linear baselines.

In [ ]:
df_res = pd.DataFrame({k: {m: results[k][m] for m in ['RMSE ($)','MAE ($)','R²','CV R² mean','CV R² std','Train Time (s)']}
                       for k in results}).T

model_colors = plt.cm.tab10(np.linspace(0, 0.9, len(models)))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics_plot = [('RMSE ($)', 'RMSE (USD)', 'lower is better'),
                ('MAE ($)',  'MAE (USD)',  'lower is better'),
                ('R²',       'R² Score',  'higher is better')]

for ax, (col, ylabel, note) in zip(axes, metrics_plot):
    vals   = df_res[col].astype(float)
    bars   = ax.barh(vals.index, vals.values,
                     color=model_colors, edgecolor='white', height=0.6)
    ax.set_xlabel(f'{ylabel}\n({note})', fontsize=LABEL_FS)
    ax.set_title(ylabel, fontsize=13, fontweight='bold')
    if col == 'R²':
        ax.set_xlim(0, 1.05)
    for bar, val in zip(bars, vals.values):
        fmt = f'{val:.3f}' if col == 'R²' else f'${val:,.0f}'
        ax.text(val * 1.01, bar.get_y() + bar.get_height()/2,
                fmt, va='center', fontsize=8)

fig.suptitle('Model Comparison: Test-Set Performance', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../plot05_01_model_comparison_metrics.png', dpi=FIG_DPI)
plt.show()
df_res.drop(columns=['CV R² std']).round(4)

---
### 📊 PLOT 2 — Cross-Validation R² with Error Bars
**What it shows:** 5-fold CV mean R² ± 1 std for each model.  
**Presentation use:** Shows stability/generalisability, not just single split performance; small error bars = consistent model.

In [ ]:
cv_means = df_res['CV R² mean'].astype(float)
cv_stds  = df_res['CV R² std'].astype(float)

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(cv_means))
bars = ax.barh(y_pos, cv_means.values, xerr=cv_stds.values,
               color=model_colors, edgecolor='white',
               error_kw=dict(ecolor='black', capsize=5, capthick=1.5, elinewidth=1.5),
               height=0.55)
ax.set_yticks(y_pos)
ax.set_yticklabels(cv_means.index, fontsize=11)
ax.set_xlabel('Cross-Validation R² (5-fold, higher is better)', fontsize=LABEL_FS)
ax.set_xlim(0, 1.1)
ax.set_title('5-Fold CV R² ± Std Dev\n(error bars show variance across folds)',
             fontsize=TITLE_FS, fontweight='bold')
for bar, val, std in zip(bars, cv_means.values, cv_stds.values):
    ax.text(val + std + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}±{std:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../plot05_02_cv_r2_errorbars.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 3 — Training Time vs R² Trade-off
**What it shows:** Scatter of training time vs test R², bubble size = RMSE.  
**Presentation use:** Practical engineering slide — shows that Random Forest / Gradient Boosting cost more compute but deliver much better accuracy.

In [ ]:
times = df_res['Train Time (s)'].astype(float)
r2s   = df_res['R²'].astype(float)
rmses = df_res['RMSE ($)'].astype(float)

fig, ax = plt.subplots(figsize=(9, 6))
for i, name in enumerate(df_res.index):
    size = (rmses[name] / rmses.max()) * 800 + 80
    ax.scatter(times[name], r2s[name], s=size, color=model_colors[i],
               alpha=0.8, edgecolors='white', linewidth=1.5, zorder=5)
    ax.annotate(name, (times[name], r2s[name]),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.set_xlabel('Training Time (seconds)', fontsize=LABEL_FS)
ax.set_ylabel('Test R²', fontsize=LABEL_FS)
ax.set_title('Training Time vs Accuracy Trade-off\n(bubble size ∝ RMSE — smaller bubble = lower error)',
             fontsize=TITLE_FS, fontweight='bold')
plt.tight_layout()
plt.savefig('../plot05_03_time_vs_r2.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 4 — Ridge/Lasso Regularisation Path
**What it shows:** How test RMSE changes as regularisation strength α increases for Ridge and Lasso.  
**Presentation use:** Explains why tuning regularisation matters and shows the optimal α range.

In [ ]:
alphas = np.logspace(-4, 4, 50)
ridge_rmse, lasso_rmse = [], []

for a in alphas:
    r = Ridge(alpha=a).fit(X_train, y_train)
    ridge_rmse.append(np.sqrt(mean_squared_error(y_test, r.predict(X_test))) * SCALE)

    l = Lasso(alpha=a/SCALE, max_iter=5000).fit(X_train, y_train)
    lasso_rmse.append(np.sqrt(mean_squared_error(y_test, l.predict(X_test))) * SCALE)

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(alphas, ridge_rmse, 'b-o', markersize=4, label='Ridge')
ax.semilogx(alphas, lasso_rmse, 'r-s', markersize=4, label='Lasso')
ax.axvline(1.0, color='blue', linestyle='--', alpha=0.5, label='Ridge α=1 (used)')
ax.set_xlabel('Regularisation α (log scale)', fontsize=LABEL_FS)
ax.set_ylabel('Test RMSE (USD)', fontsize=LABEL_FS)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Regularisation Path: Ridge vs Lasso',
             fontsize=TITLE_FS, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../plot05_04_regularisation_path.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 5 — Decision Tree: max_depth Tuning
**What it shows:** Train vs test R² as tree depth increases.  
**Presentation use:** Classic overfitting illustration — train score keeps rising but test score peaks then degrades.

In [ ]:
depths = range(1, 25)
train_r2, test_r2 = [], []

for d in depths:
    dt = DecisionTreeRegressor(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_r2.append(r2_score(y_train, dt.predict(X_train)))
    test_r2.append(r2_score(y_test,  dt.predict(X_test)))

best_depth = depths[np.argmax(test_r2)]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(depths, train_r2, 'b-o', markersize=5, label='Train R²')
ax.plot(depths, test_r2,  'r-s', markersize=5, label='Test R²')
ax.axvline(best_depth, color='green', linestyle='--', linewidth=1.5,
           label=f'Best depth = {best_depth}')
ax.set_xlabel('max_depth', fontsize=LABEL_FS)
ax.set_ylabel('R² Score', fontsize=LABEL_FS)
ax.set_title('Decision Tree: Overfitting vs Depth',
             fontsize=TITLE_FS, fontweight='bold')
ax.legend(fontsize=10)
ax.fill_between(depths, train_r2, test_r2, alpha=0.08, color='red', label='Generalisation gap')
plt.tight_layout()
plt.savefig('../plot05_05_dt_depth_tuning.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 6 — Learning Curves (Random Forest)
**What it shows:** Training and CV score as the number of training samples increases.  
**Presentation use:** Shows whether the model benefits from more data — the gap between curves tells you if you're in a bias or variance regime.

In [ ]:
rf_model = results['Random Forest']['model']

train_sizes, train_scores, val_scores = learning_curve(
    rf_model, X_train, y_train,
    cv=5, scoring='r2',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

tr_mean, tr_std = train_scores.mean(axis=1), train_scores.std(axis=1)
va_mean, va_std = val_scores.mean(axis=1),   val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, tr_mean, 'b-o', markersize=5, label='Training R²')
ax.fill_between(train_sizes, tr_mean-tr_std, tr_mean+tr_std, alpha=0.15, color='blue')
ax.plot(train_sizes, va_mean, 'r-s', markersize=5, label='CV R²')
ax.fill_between(train_sizes, va_mean-va_std, va_mean+va_std, alpha=0.15, color='red')
ax.set_xlabel('Training Set Size (samples)', fontsize=LABEL_FS)
ax.set_ylabel('R² Score', fontsize=LABEL_FS)
ax.set_title('Learning Curves — Random Forest',
             fontsize=TITLE_FS, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../plot05_06_learning_curves_rf.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 7 — Metric Comparison Radar Chart
**What it shows:** Spider/radar chart placing each model on normalised RMSE, MAE, R², and CV R² axes.  
**Presentation use:** Visually compelling one-page summary of model trade-offs for an executive audience.

In [ ]:
radar_metrics = ['R²', 'CV R² mean']
# For RMSE and MAE: invert (1 - normalised) so larger = better
df_radar = df_res[['RMSE ($)','MAE ($)','R²','CV R² mean']].astype(float).copy()
df_radar['RMSE_inv'] = 1 - (df_radar['RMSE ($)'] - df_radar['RMSE ($)'].min()) / \
                           (df_radar['RMSE ($)'].max() - df_radar['RMSE ($)'].min())
df_radar['MAE_inv']  = 1 - (df_radar['MAE ($)']  - df_radar['MAE ($)'].min())  / \
                           (df_radar['MAE ($)'].max()  - df_radar['MAE ($)'].min())
plot_cols = ['RMSE_inv','MAE_inv','R²','CV R² mean']
col_labels = ['RMSE\n(inverted)','MAE\n(inverted)','R²','CV R²']

N      = len(plot_cols)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = plt.cm.tab10(np.linspace(0, 0.9, len(df_radar)))

for i, (idx, row) in enumerate(df_radar[plot_cols].iterrows()):
    vals_r = row.tolist() + [row.iloc[0]]
    ax.plot(angles, vals_r, linewidth=2, color=colors[i], label=idx)
    ax.fill(angles, vals_r, alpha=0.07, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(col_labels, fontsize=11)
ax.set_title('Model Performance Radar\n(all axes: larger = better)',
             fontsize=TITLE_FS, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15), fontsize=9)
plt.tight_layout()
plt.savefig('../plot05_07_model_radar.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 8 — Performance Summary Heatmap (Table)
**What it shows:** Colour-coded metric table — green = best, red = worst.  
**Presentation use:** Quick, data-dense slide for technical audiences who want numbers and colour cues at once.

In [ ]:
summary_df = df_res[['RMSE ($)','MAE ($)','R²','CV R² mean','Train Time (s)']].astype(float)

# Normalise each column 0-1; for RMSE/MAE/time: invert (lower = greener)
norm = summary_df.copy()
for col in ['RMSE ($)','MAE ($)','Train Time (s)']:
    norm[col] = 1 - (summary_df[col] - summary_df[col].min()) / \
                    (summary_df[col].max() - summary_df[col].min())
for col in ['R²','CV R² mean']:
    norm[col] = (summary_df[col] - summary_df[col].min()) / \
                (summary_df[col].max() - summary_df[col].min())

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    norm, annot=summary_df.round(3), fmt='g',
    cmap='RdYlGn', vmin=0, vmax=1,
    linewidths=0.5, ax=ax, cbar=False,
    annot_kws={'size': 10}
)
ax.set_title('Model Performance Heatmap\n(green = best in column, RMSE/MAE/Time: lower is better)',
             fontsize=TITLE_FS, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
plt.savefig('../plot05_08_performance_heatmap.png', dpi=FIG_DPI)
plt.show()

---
## 2 · Save Best Model

In [ ]:
best_name = df_res['R²'].astype(float).idxmax()
best_model = results[best_name]['model']
print(f'Best model: {best_name}  (R²={results[best_name]["R²"]:.4f})')

os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_model.pkl')
print('Saved to ../models/best_model.pkl')

# Save all results for notebook 06
np.save('../../data/processed/y_pred_best.npy', results[best_name]['y_pred'])

# Also save RF predictions for comparison
np.save('../../data/processed/y_pred_rf.npy',  results['Random Forest']['y_pred'])
np.save('../../data/processed/y_pred_gb.npy',  results['Gradient Boosting']['y_pred'])
np.save('../../data/processed/y_pred_lr.npy',  results['Linear Regression']['y_pred'])

---
## Summary of Notebook 05

| Plot | File | Key Takeaway |
|------|------|--------------|
| 1 | plot05_01_model_comparison_metrics | Gradient Boosting / RF best on RMSE, MAE, R² |
| 2 | plot05_02_cv_r2_errorbars | Ensemble models are stable across folds |
| 3 | plot05_03_time_vs_r2 | Linear models fast but inaccurate; ensemble worth the cost |
| 4 | plot05_04_regularisation_path | Optimal α for Ridge around 1-10 |
| 5 | plot05_05_dt_depth_tuning | Classic overfit curve; depth=8 is sweet spot |
| 6 | plot05_06_learning_curves_rf | RF still improving — more data would help |
| 7 | plot05_07_model_radar | Visual comparison spider chart |
| 8 | plot05_08_performance_heatmap | At-a-glance colour-coded score table |